In [1]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline, PruneGraph
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
# from circuit_tracer.subgraph.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [4]:
explainer = shap.Explainer(model, tokenizer)

In [5]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [ ]:
shap_values = explainer(prompts)

In [ ]:
shap.plots.text(shap_values)

In [52]:
print(shap_values)

.values =
array([[[-4.22446732e-06],
        [ 6.54286429e+00],
        [ 1.68508452e+00],
        [ 1.20230146e+00],
        [ 6.04736818e+00],
        [ 6.77137808e-02],
        [ 4.40673760e+00],
        [ 4.12085123e-01],
        [ 4.40843961e-01]]])

.base_values =
array([[-21.73993724]])

.data =
(array(['', 'Cat', ' is', ' to', ' kitten', ' as', ' dog', ' is', ' to'],
      dtype=object),)


In [34]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [4]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="austin_plt",
    source_set = 'gemmascope-transcoder-16k',
    out_path="temp_graph_files/austin_plt.json",
    node_threshold = 0.95,
    edge_threshold = 0.98,
)

requesting graph for slug='austin_plt' prompt='Fact: The capital of the state containing Dallas is...'


downloading graph json from https://neuronpedia-attrib.s3.us-east-1.amazonaws.com/user-graphs/anonymous/austin_plt.json
saved austin_plt -> temp_graph_files/austin_plt.json (nodes=5387)


## ASG Pipeline

### Prune

In [3]:
import circuit_tracer, circuit_tracer.subgraph.prune as p, circuit_tracer.graph as g; print(circuit_tracer.__file__); print(p.__file__); print(g.__file__)

d:\anaconda\envs\circuit\Lib\site-packages\circuit_tracer\__init__.py
d:\anaconda\envs\circuit\Lib\site-packages\circuit_tracer\subgraph\prune.py
d:\anaconda\envs\circuit\Lib\site-packages\circuit_tracer\graph.py


In [1]:
from circuit_tracer.subgraph.prune import prune_graph_pipeline
from circuit_tracer.subgraph.cluster import cluster_graph, cluster_graph_with_labels
from circuit_tracer.subgraph.auto_grouping import find_best_k

# prune_graph = load_prune_graph('subgraph/puppy_clt_shap_07_085.pt')
token_weights = [0,0,0,0,1/3,0,0,1/3,0,1/3,0]
# 1) Build prune_graph as usual
prune_graph = prune_graph_pipeline(
    json_path="temp_graph_files/austin_clt.json",
    token_weights = token_weights,
    logit_weights="target",
    combined_scores_method="geometric",
    node_threshold=0.5,
    edge_threshold=0.95,
    alpha=0.55,
    keep_all_tokens_and_logits=False,
    filter_act_density=False,
)


In [2]:
prune_graph.num_nodes, prune_graph.num_edges

(24, 122)

In [3]:
for node_id in prune_graph.kept_ids:
    print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_scores[prune_graph.kept_ids.index(node_id)].item())


1_89326_9 Dallas 0.010695972479879856
1_72774_10 Code/licensing snippets 0.011104445904493332
7_24743_9 Texas legal matters 0.010370000265538692
16_89970_9 Texas 0.014714894816279411
17_1822_10 place names 0.012354392558336258
17_98126_10 Government buildings/institutions 0.010455240495502949
18_3623_10 capital cities 0.015131400898098946
20_44686_9 Texas locations/legal contexts 0.02104082703590393
20_44686_10 Texas locations/legal contexts 0.027731915935873985
20_74108_10 capital 0.01630185917019844
21_16875_10 cities 0.012499647215008736
21_61721_10 code/documentation 0.013867194764316082
21_84338_10 Cities and locations 0.019802715629339218
22_11998_10 Texas and Dallas 0.022554252296686172
22_32893_10 Texas legal documents 0.01790972799062729
22_81571_10 Texas locations and organizations 0.011023018509149551
23_31673_10 Court appeals at specific location 0.011793253943324089
23_59746_10 Technical/Scientific writing 0.010731072165071964
24_5303_10 legal documents 0.01062100660055875

In [4]:
print(prune_graph.node_scores.max(), prune_graph.node_scores.min())
print(torch.sum(prune_graph.node_scores))

tensor(0.5892) tensor(0.0104)


NameError: name 'torch' is not defined

In [5]:
from circuit_tracer.subgraph.prune import save_prune_graph, load_prune_graph
save_prune_graph(prune_graph, "subgraph/austin_plt_clean_45_55.pt")
# loaded_prune_graph = load_prune_graph("temp_graph_files/austin_clt_")

In [5]:
# 2) Auto-select k
best_k, sweep = find_best_k(
    prune_graph,
    max_layer_span=4,
    gamma = 1,
    # optional knobs:
    # k_min_override=3,
    # k_max_override=12,
    # max_sn=10,
    # weights={"w_intra": 0.3, "w_dag": 0.25, "w_flow": 0.25, "w_size": 0.2},
)

print("best_k:", best_k)
print("best score:", sweep[best_k]["total"] if best_k in sweep else None)

# 3) Run final clustering with best_k
supernodes = cluster_graph_with_labels(
    prune_graph,
    target_k=11,
    max_layer_span=4,
)

ValueError: operands could not be broadcast together with shapes (20,20,24) (24,20,20) 

In [12]:
supernodes

[['cluster_1', '14_2268_9', '16_25_9', '18_1437_10', '18_6101_10'],
 ['cluster_2', '18_8959_10', '19_1445_10', '19_2695_10', '19_7477_10'],
 ['cluster_3', '20_15589_9', '20_15589_10'],
 ['cluster_6', '22_3551_10', '22_4999_10'],
 ['cluster_7', '22_11718_10', '23_11444_10', '23_12237_10', '23_13193_10'],
 ['cluster_8', '23_13841_10', '23_15366_10', '24_6044_10', '24_6394_10'],
 ['cluster_9', '24_13277_10', '24_15694_10', '25_553_10', '25_583_10'],
 ['cluster_10', '25_4259_10', '25_4679_10'],
 ['cluster_11', '25_4717_10', '25_4886_10'],
 ['cluster_12', '25_9975_10', '25_13300_10']]

In [13]:
for supernode in supernodes:
    print(supernode[0])
    for node_id in supernode[1:]:
        print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_scores[prune_graph.kept_ids.index(node_id)].item())

cluster_1
14_2268_9 Texas 0.014798968099057674
16_25_9 Texas legal documents 0.018482232466340065
18_1437_10 Legal documents from Texas 0.011999973095953465
18_6101_10 capital cities 0.014354558661580086
cluster_2
18_8959_10 government/state 0.01550628524273634
19_1445_10 Downtowns of cities 0.011983275413513184
19_2695_10 cities 0.015295961871743202
19_7477_10 Dallas 0.01141655258834362
cluster_3
20_15589_9 Texas 0.027857357636094093
20_15589_10 Texas 0.05035414546728134
cluster_6
22_3551_10 Place names and legal cases 0.010933110490441322
22_4999_10 Locations 0.014251688495278358
cluster_7
22_11718_10 Texas locations 0.015039621852338314
23_11444_10 cities and places 0.011511438526213169
23_12237_10 Cities and states names 0.03035122901201248
23_13193_10 Legal and Southern place names 0.012916097417473793
cluster_8
23_13841_10 towns and cities 0.01188632845878601
23_15366_10 Code snippets 0.01146793644875288
24_6044_10 in 0.05099157989025116
24_6394_10 locations 0.028498627245426178


In [7]:
import json
from typing import List
def extract_supernodes(flow_map_path: str) -> List[List[str]]:
    """
    Load a mapping JSON like flow_analysis_supernode_map.json and return
    a list of supernodes in the format:
      [ ["SN_LABEL", "node_a", "node_b", ...], ... ]
    """
    p = Path(flow_map_path)
    with p.open("r", encoding="utf-8") as f:
        mapping = json.load(f)

    supernodes = [[label, *nodes] for label, nodes in mapping.items()]
    return supernodes

### Visualize on the web

In [23]:
import json
from circuit_tracer.subgraph.api import save_subgraph

# Expected format for save_subgraph:
# supernodes = [
#   ["label_1", "node_id_a", "node_id_b"],
#   ["label_2", "node_id_c", "node_id_d", "node_id_e"],
# ]
# supernodes = extract_supernodes("flow_analysis_supernode_map.json")
# print(supernodes)


In [41]:
status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug="factthecapitalof-1771775020852",                       # parent graph slug
    displayName="austin-grouped-shap",     # subgraph name shown on Neuronpedia
    pinnedIds=prune_graph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.7,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

{'Content-Type': 'application/json', 'x-api-key': 'sk-np-Vi0OcQl8zNm9jC7nyGRYfwssvSakMyrPjV675uEWIuU0'}


status: 200
{'success': True, 'subgraphId': 'cmnybzrgv0002uz3gx86q78xb'}


# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality